In [5]:
import os
from typing import Optional
from datetime import datetime

# ==========================================
# 1. LLM Setup
# ==========================================
from langchain_ollama import ChatOllama

print("🔄 Initializing Ollama (Llama 3.1)...")
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.3,
    timeout=60  
)

# ==========================================
# 2. Vector Database Setup
# ==========================================
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

retriever = vectordb.as_retriever(
    search_kwargs={"k": 4}
)

# ==========================================
# 3. Query Rewriting Setup
# ==========================================
from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_template(
    """Based on the conversation history, rewrite the user's question to be more specific and searchable for a vector database.

Conversation History:
{history}

Current Question:
{question}

Return ONLY the improved search query, nothing else."""
)

rewrite_chain = rewrite_prompt | llm

# ==========================================
# 4. Tool Definitions
# ==========================================
from langchain_core.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun

# FIX 1: Corrected instantiation of DuckDuckGo tool
wikipedia = WikipediaAPIWrapper()
ddg_search = DuckDuckGoSearchRun()

@tool
def knowledge_base_search(query: str) -> str:
    """
    Search the internal knowledge base (vector database).
    Improves the query for better vector DB search.
    Returns relevant documents or empty if no matches found.
    """
    try:
        improved_query = rewrite_chain.invoke(
            {
                "history": "No prior conversation history",
                "question": query
            }
        ).content.strip()
        
        print(f"\n📝 Original Query: {query}")
        print(f"🔄 Rewritten Query: {improved_query}")
        
        docs = retriever.invoke(improved_query)
        
        if docs:
            result = "\n\n".join(
                f"[Source: {doc.metadata.get('source', 'Unknown')}]\n{doc.page_content}"
                for doc in docs
            )
            print(f"✅ Found {len(docs)} relevant documents in knowledge base")
            return result
        else:
            print("⚠️ No relevant documents found in knowledge base")
            return ""
            
    except Exception as e:
        print(f"❌ Error searching knowledge base: {str(e)}")
        return ""


@tool
def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia for information.
    Use this when knowledge base doesn't have relevant information.
    """
    try:
        print(f"\n📚 Searching Wikipedia for: {query}")
        result = wikipedia.run(query)
        if result and result.strip():
            print(f"✅ Found Wikipedia results")
            return f"Wikipedia Results:\n{result}"
        else:
            print("⚠️ No Wikipedia results found")
            return ""
    except Exception as e:
        print(f"❌ Error searching Wikipedia: {str(e)}")
        return ""


@tool
def web_search_duckduckgo(query: str) -> str:
    """
    Search the web using DuckDuckGo.
    Use this when knowledge base and Wikipedia don't have relevant information.
    """
    try:
        print(f"\n🌐 Searching DuckDuckGo for: {query}")
        result = ddg_search.run(query)
        if result and result.strip():
            print(f"✅ Found DuckDuckGo results")
            return f"Web Results:\n{result}"
        else:
            print("⚠️ No web results found")
            return ""
    except Exception as e:
        print(f"❌ Error searching DuckDuckGo: {str(e)}")
        return ""


@tool
def rewrite_query_with_history(question: str, conversation_history: str) -> str:
    """
    Rewrite a question based on conversation history for better search results.
    This tool helps improve queries by considering the context of previous messages.
    """
    try:
        improved_query = rewrite_chain.invoke(
            {
                "history": conversation_history,
                "question": question
            }
        ).content.strip()
        
        print(f"\n📝 Query Rewrite:")
        print(f"  Original: {question}")
        print(f"  Improved: {improved_query}")
        
        return improved_query
    except Exception as e:
        print(f"❌ Error rewriting query: {str(e)}")
        return question


# ==========================================
# 5. Agent Setup
# ==========================================
# NEW & WORKING
from langchain_classic.agents import AgentExecutor
from langgraph.prebuilt import create_react_agent
from langchain_classic import hub

tools = [
    knowledge_base_search,
    wikipedia_search,
    web_search_duckduckgo,
    rewrite_query_with_history
]

# FIX 2: Pulled official ReAct template and appended your routing custom behavior 
base_prompt = hub.pull("hwchase17/react")

system_instructions = """You are an intelligent RAG (Retrieval-Augmented Generation) assistant.

Your workflow:
1. When a user asks a question, first use 'rewrite_query_with_history' if there's conversation context
2. Always start by searching the 'knowledge_base_search' (internal documents first)
3. If knowledge base returns empty or insufficient results, search 'wikipedia_search'
4. If still no results, use 'web_search_duckduckgo' for broader web information
5. Synthesize all findings into a comprehensive answer

Important:
- Always try knowledge base first - it's the most reliable source
- Check if results are actually relevant before using them
- If you find good information, don't search further
- Combine information from multiple sources when appropriate
- Be clear about your information sources

When responding:
- Always cite where you found the information
- Be specific and helpful
- If you can't find relevant information, be honest about it

"""

# Prefix instructions cleanly into standard ReAct formatting structure
agent_prompt = base_prompt.partial(
    tools=tools,
    chat_history=""  # Placeholder to keep the prompt happy if needed
)
agent_prompt.template = system_instructions + "\nChat History Context:\n{chat_history}\n\n" + base_prompt.template

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=agent_prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
)

# ==========================================
# 6. Conversation Management
# ==========================================
class ConversationManager:
    """Manages conversation history for context-aware search"""
    
    def __init__(self, max_history: int = 5):
        self.history = []
        self.max_history = max_history
    
    def add_message(self, role: str, content: str):
        """Add a message to conversation history"""
        self.history.append({
            "role": role,
            "content": content,
            "timestamp": datetime.now()
        })
        
        if len(self.history) > self.max_history * 2:
            self.history = self.history[-self.max_history*2:]
    
    def get_history_string(self) -> str:
        """Get conversation history as formatted string"""
        if not self.history:
            return "No prior conversation"
        
        history_str = ""
        for msg in self.history[-self.max_history:]:
            history_str += f"{msg['role'].upper()}: {msg['content']}\n"
        
        return history_str
    
    def clear(self):
        """Clear conversation history"""
        self.history = []


# ==========================================
# 7. Main RAG Agent Interface
# ==========================================
class RAGAgent:
    """Main interface for the RAG agent system"""
    
    def __init__(self):
        self.conversation_manager = ConversationManager()
    
    def query(self, question: str) -> str:
        """Process a user question through the RAG system"""
        print(f"\n{'='*60}")
        print(f"🤖 Processing Question: {question}")
        print(f"{'='*60}")
        
        self.conversation_manager.add_message("user", question)
        chat_history = self.conversation_manager.get_history_string()
        
        try:
            response = agent_executor.invoke({
                "input": question,
                "chat_history": chat_history
            })
            
            answer = response.get("output", "No response generated")
            self.conversation_manager.add_message("assistant", answer)
            
            print(f"\n✅ Final Answer:")
            print(f"{answer}")
            
            return answer
            
        except Exception as e:
            error_msg = f"❌ Error processing query: {str(e)}"
            print(error_msg)
            self.conversation_manager.add_message("assistant", error_msg)
            return error_msg
    
    def chat(self):
        """Interactive chat mode"""
        print("\n" + "="*60)
        print("🚀 RAG Agent - Interactive Mode")
        print("="*60)
        print("Type 'exit' to quit, 'clear' to clear history\n")
        
        while True:
            try:
                question = input("\n📝 You: ").strip()
                
                if not question:
                    continue
                elif question.lower() == "exit":
                    print("\n👋 Goodbye!")
                    break
                elif question.lower() == "clear":
                    self.conversation_manager.clear()
                    print("🔄 Conversation history cleared")
                    continue
                
                self.query(question)
                
            except KeyboardInterrupt:
                print("\n\n👋 Interrupted. Goodbye!")
                break
            except Exception as e:
                print(f"❌ Error: {str(e)}")

# if __name__ == "__main__":
#     rag_agent = RAGAgent()
#     rag_agent.chat()

🔄 Initializing Ollama (Llama 3.1)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7598.51it/s]
C:\Users\Administrator\AppData\Local\Temp\ipykernel_29208\2592173089.py:183: LangChainDeprecationWarning: langchain_classic.hub.pull is deprecated. Use the LangSmith SDK instead.
  base_prompt = hub.pull("hwchase17/react")


ValueError: Pulling a public prompt by owner/name is disabled by default because prompts may contain untrusted serialized LangChain objects. If you trust this prompt, set `dangerously_pull_public_prompt=True` to acknowledge the risk.

In [ ]:
# ==========================================
# 8. Example Usage
# ==========================================
if __name__ == "__main__":
    # Initialize the RAG agent
    rag_agent = RAGAgent()
    
    # Example 1: Single query
    print("\n" + "🎯 Example 1: Single Query")
    print("="*60)
    answer = rag_agent.query("What is machine learning?")
    
    # Example 2: Follow-up question (uses conversation history)
    print("\n" + "🎯 Example 2: Follow-up Question")
    print("="*60)
    answer = rag_agent.query("Tell me more about supervised learning")
    
    # Example 3: Interactive chat
    print("\n" + "🎯 Example 3: Start Interactive Chat")
    print("="*60)

In [2]:
from langchain_classic.agents import AgentExecutor